In [1]:
# --- 1. LIBRARIES ---
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning & Scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, roc_curve, auc)

# Deep Learning (TensorFlow/Keras)
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (Dense, Conv1D, BatchNormalization, 
                                     concatenate, Flatten, Dropout, Attention, 
                                     Reshape, GlobalAveragePooling1D, Multiply, MaxPooling1D)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
# --- 2. REPRODUCIBILITY (The "Same Result" Rule) ---
def set_seeds(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)
    print(f"Environment seeds locked at {seed}")

set_seeds(42)

Environment seeds locked at 42


In [3]:
# Load the specific binary dataset
df = pd.read_csv("CIC-DDoS-2019_training.csv")

# Verify the load by checking the total rows and columns
print(f"Dataset successfully loaded.")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")

Dataset successfully loaded.
Total Rows: 50063112
Total Columns: 84


In [4]:
# 1. Strip any leading/trailing whitespace and convert to lowercase
df.columns = df.columns.str.strip().str.lower()

# 2. Replace spaces, slashes, and dots with underscores for easy coding
df.columns = df.columns.str.replace(' ', '_', regex=False)
df.columns = df.columns.str.replace('/', '_', regex=False)
df.columns = df.columns.str.replace('.', '_', regex=False)

# 3. Print the first 10 columns to verify the change
print("First 10 standardized columns:")
print(df.columns[:10].tolist())

# 4. Find the exact name of your Label column
label_col = [col for col in df.columns if 'label' in col]
print(f"\nLabel column found: {label_col}")

First 10 standardized columns:
['flow_id', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'protocol', 'timestamp', 'flow_duration', 'tot_fwd_pkts', 'tot_bwd_pkts']

Label column found: ['label']


In [5]:
# List of columns that are not useful for training features
identifiers = ['flow_id', 'src_ip', 'dst_ip', 'timestamp']

# Drop them only if they exist in the current dataframe
df.drop(columns=[col for col in identifiers if col in df.columns], inplace=True)

print(f"Identifiers removed.")
print(f"Remaining columns: {df.shape[1]}")

Identifiers removed.
Remaining columns: 80


In [6]:
import joblib
import numpy as np

imputer_bundle = joblib.load("regression_imputer.pkl")

imputer_models = imputer_bundle['models']
COLS_WONA = imputer_bundle['cols_wona']
COLS_WNA = imputer_bundle['cols_wna']


In [7]:
# Replace Inf with NaN (same as training)
df.replace([np.inf, -np.inf], np.nan, inplace=True)


In [8]:
print("Remaining NaNs:", df.isna().sum().sum())


Remaining NaNs: 2726472


In [9]:
for target_col, model in imputer_models.items():
    if target_col not in df.columns:
        continue  # safety check

    predict_data = df[df[target_col].isna()]

    if not predict_data.empty:
        # Ensure predictor columns exist
        X = predict_data[COLS_WONA]

        predictions = model.predict(X)
        df.loc[df[target_col].isna(), target_col] = predictions


In [10]:
print("Remaining NaNs:", df.isna().sum().sum())


Remaining NaNs: 0


In [11]:
# 1. Check for any columns that are not integers or floats (excluding 'label')
non_numeric = df.drop(columns=['label']).select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Non-numeric columns found (excluding label): {non_numeric}")

# 2. Check for Infinite values (common in this dataset)
inf_count = np.isinf(df.select_dtypes(include=[np.number])).values.sum()
print(f"Total Infinite values: {inf_count}")

# 3. Check for Null/NaN values
nan_count = df.isnull().sum().sum()
print(f"Total NaN values: {nan_count}")

Non-numeric columns found (excluding label): []
Total Infinite values: 0
Total NaN values: 0


In [12]:
# 1. Check unique values in the label column
print("Unique labels in dataset:")
print(df['label'].unique())

# 2. Check the count of each label to see if it is imbalanced
print("\nLabel counts:")
print(df['label'].value_counts())

# 3. Check the data type of the label
print(f"\nLabel data type: {df['label'].dtype}")

Unique labels in dataset:
['DrDoS_DNS' 'BENIGN' 'DrDoS_LDAP' 'DrDoS_MSSQL' 'DrDoS_NetBIOS'
 'DrDoS_NTP' 'DrDoS_SNMP' 'DrDoS_SSDP' 'DrDoS_UDP' 'Syn' 'TFTP' 'UDP-lag'
 'WebDDoS']

Label counts:
label
TFTP             20082580
DrDoS_SNMP        5159870
DrDoS_DNS         5071011
DrDoS_MSSQL       4522492
DrDoS_NetBIOS     4093279
DrDoS_UDP         3134645
DrDoS_SSDP        2610611
DrDoS_LDAP        2179930
Syn               1582289
DrDoS_NTP         1202642
UDP-lag            366461
BENIGN              56863
WebDDoS               439
Name: count, dtype: int64

Label data type: object


In [13]:
# 1. Define Features (X) and Target (y)
X = df.drop(columns=['label'])
y = df['label']

# 2. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Verification of the split
print(f"Training set: {X_train.shape[0]} rows")
print(f"Testing set:  {X_test.shape[0]} rows")

Training set: 40050489 rows
Testing set:  10012623 rows


In [14]:
# 1. Initialize the Scaler
scaler = MinMaxScaler()

# 2. Fit on training data and transform both sets
# This preserves the default float64 precision
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Verification of the results
print(f"Scaling Complete.")
print(f"Data Type:         {X_train_scaled.dtype}")
print(f"Number of Features: {X_train_scaled.shape[1]}")
print(f"Value Range:       {X_train_scaled.min()} to {X_train_scaled.max()}")

Scaling Complete.
Data Type:         float64
Number of Features: 79
Value Range:       0.0 to 1.0


In [15]:
import joblib

joblib.dump(scaler, "scaler.pkl")
print("Scaler saved to scaler.pkl")


Scaler saved to scaler.pkl


In [16]:
# --- STAGE 1: AUTOENCODER DEFINITION ---

# 1. Use your 79 features as the input dimension
input_dim = X_train_scaled.shape[1]
ae_in = Input(shape=(input_dim,))

# 2. Encoder: Bottleneck strategy (96 neurons -> 32 neurons)
enc = Dense(96, activation='relu')(ae_in)
enc = BatchNormalization()(enc)
enc = Dense(32, activation='relu')(enc) 

# 3. AE Attention Mechanism
# Reshaping to (1, 32) allows the Attention layer to weigh the encoded features
att_reshape = Reshape((1, 32))(enc)
att_logic = Attention()([att_reshape, att_reshape])
att_flat = Flatten()(att_logic)

# 4. Decoder: Reconstruction strategy
# Maps the 32 features back to the original 79 dimensions
dec = Dense(96, activation='tanh')(att_flat)
dec_out = Dense(input_dim, activation='sigmoid')(dec) 

# 5. Model Creation
# 'autoencoder' is for training; 'encoder_only' is for feature extraction
autoencoder = Model(ae_in, dec_out, name="Autoencoder")
encoder_only = Model(ae_in, att_flat, name="Encoder_Extractor")

# 6. Compilation
# Using Adam optimizer and Mean Absolute Error (MAE) loss
autoencoder.compile(optimizer='adam', loss='mae')

# Display the architecture
autoencoder.summary()

Model: "Autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 79)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 96)                │           7,680 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 96)                │             384 │ dense[0][0]                │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 32)                │           3,104 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ reshape (Reshape)             │ (None, 1, 32)             │               0 │ dense_1[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ attention (Attention)         │ (None, 1, 32)             │               0 │ reshape[0][0],             │
│                               │                           │                 │ reshape[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten (Flatten)             │ (None, 32)                │               0 │ attention[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 96)                │           3,168 │ flatten[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_3 (Dense)               │ (None, 79)                │           7,663 │ dense_2[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 21,999 (85.93 KB)

 Trainable params: 21,807 (85.18 KB)

 Non-trainable params: 192 (768.00 B)

In [17]:
# 1. Define Early Stopping to monitor validation loss
early_stop_ae = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

# 2. Train the model
# Input (X_train_scaled) is also the Target (X_train_scaled) for an Autoencoder
print("Starting Autoencoder Training...")
history_ae = autoencoder.fit(
    X_train_scaled, X_train_scaled,
    epochs=30,
    batch_size=1024,
    validation_split=0.2,
    callbacks=[early_stop_ae],
    verbose=1
)

print("\nAutoencoder training finished.")

Starting Autoencoder Training...
Epoch 1/30


C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\ops\nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


31290/31290 ━━━━━━━━━━━━━━━━━━━━ 226s 5ms/step - loss: 0.0018 - val_loss: 0.0012
Epoch 2/30
31290/31290 ━━━━━━━━━━━━━━━━━━━━ 148s 5ms/step - loss: 9.1562e-04 - val_loss: 8.2089e-04
Epoch 3/30
31290/31290 ━━━━━━━━━━━━━━━━━━━━ 147s 5ms/step - loss: 8.4850e-04 - val_loss: 8.4598e-04
Epoch 4/30
31290/31290 ━━━━━━━━━━━━━━━━━━━━ 151s 5ms/step - loss: 8.0118e-04 - val_loss: 8.7837e-04
Epoch 5/30
31290/31290 ━━━━━━━━━━━━━━━━━━━━ 146s 5ms/step - loss: 7.6968e-04 - val_loss: 6.7800e-04
Epoch 6/30
31290/31290 ━━━━━━━━━━━━━━━━━━━━ 148s 5ms/step - loss: 7.4690e-04 - val_loss: 7.7965e-04
Epoch 7/30
31290/31290 ━━━━━━━━━━━━━━━━━━━━ 143s 5ms/step - loss: 7.3021e-04 - val_loss: 7.2037e-04
Epoch 8/30
31290/31290 ━━━━━━━━━━━━━━━━━━━━ 149s 5ms/step - loss: 7.1633e-04 - val_loss: 7.3933e-04
Epoch 9/30
31290/31290 ━━━━━━━━━━━━━━━━━━━━ 149s 5ms/step - loss: 7.0610e-04 - val_loss: 7.2768e-04
Epoch 10/30
31290/31290 ━━━━━━━━━━━━━━━━━━━━ 148s 5ms/step - loss: 6.9661e-04 - val_loss: 7.2221e-04

Autoencoder train

In [18]:
# 1. Extract the 32 compressed features using the encoder model
X_train_encoded = encoder_only.predict(X_train_scaled)
X_test_encoded = encoder_only.predict(X_test_scaled)

# 2. Reshape for the CNN (Conv1D)
# New Shape: (Samples, 32, 1)
X_train_cnn = X_train_encoded.reshape(X_train_encoded.shape[0], 32, 1)
X_test_cnn = X_test_encoded.reshape(X_test_encoded.shape[0], 32, 1)

# 3. Verification
print("Feature Extraction & Reshaping Complete.")
print(f"Original shape: {X_train_scaled.shape}")
print(f"Encoded CNN shape: {X_train_cnn.shape}")

    110/1251578 ━━━━━━━━━━━━━━━━━━━━ 19:25 932us/step

C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\ops\nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (32, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


1251578/1251578 ━━━━━━━━━━━━━━━━━━━━ 1609s 1ms/step
312895/312895 ━━━━━━━━━━━━━━━━━━━━ 2022s 6ms/step
Feature Extraction & Reshaping Complete.
Original shape: (40050489, 79)
Encoded CNN shape: (40050489, 32, 1)


In [19]:
# 2. Save the Encoder (the 32-feature extractor)
encoder_only.save("AE_Encoder_Extractor.keras")

# 3. Save the full Autoencoder (in case you want to fine-tune it later)
autoencoder.save("AE_Stage1_Full.keras")

In [20]:
# --- CNN CLASSIFIER ARCHITECTURE ---

# Input shape is (32, 1) based on the 32 features extracted by the Autoencoder
inp = Input(shape=(X_train_cnn.shape[1], 1), name="CIAM_Input")

# Inception Module: Three parallel branches with 16 filters each
conv1 = Conv1D(16, 1, activation='relu', padding='same', name="Branch_1x1")(inp)
conv3 = Conv1D(16, 3, activation='relu', padding='same', name="Branch_3x3")(inp)
conv5 = Conv1D(16, 5, activation='relu', padding='same', name="Branch_5x5")(inp)

# Merge the branches: 16 + 16 + 16 = 48 total filters
merged = concatenate([conv1, conv3, conv5], axis=-1, name="Inception_Merge")
merged = BatchNormalization(name="Merge_BN")(merged)

# Attention Mechanism: Global Average Pooling -> Dense Weighting -> Multiply
gap = GlobalAveragePooling1D(name="Attention_GAP")(merged)
att_weights = Dense(48, activation='sigmoid', name="Attention_Weights")(gap)
att_weights = Reshape((1, 48), name="Attention_Reshape")(att_weights)
merged_att = tf.keras.layers.Multiply(name="Attention_Apply")([merged, att_weights])

# Classification Head
pooled = tf.keras.layers.MaxPooling1D(pool_size=2, name="Final_Pool")(merged_att)
flat = Flatten(name="Final_Flatten")(pooled)
x = Dense(16, activation='relu', name="Dense_16")(flat)
x = Dropout(0.1, name="Dropout_1")(x)
x = Dense(8, activation='relu', name="Dense_8_A")(x)
x = Dropout(0.1, name="Dropout_2")(x)
x = Dense(8, activation='relu', name="Dense_8_B")(x)

# Output: 2 units for Softmax classification
output = Dense(2, activation='softmax', name="Output_Layer")(x)

# Create the Model object
cnn_binary = Model(inp, output, name="CIAM_Binary_Classifier")

# Display the summary to verify the layers
cnn_binary.summary()

Model: "CIAM_Binary_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ CIAM_Input (InputLayer)       │ (None, 32, 1)             │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Branch_1x1 (Conv1D)           │ (None, 32, 16)            │              32 │ CIAM_Input[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Branch_3x3 (Conv1D)           │ (None, 32, 16)            │              64 │ CIAM_Input[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Branch_5x5 (Conv1D)           │ (None, 32, 16)            │              96 │ CIAM_Input[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Inception_Merge (Concatenate) │ (None, 32, 48)            │               0 │ Branch_1x1[0][0],          │
│                               │                           │                 │ Branch_3x3[0][0],          │
│                               │                           │                 │ Branch_5x5[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Merge_BN (BatchNormalization) │ (None, 32, 48)            │             192 │ Inception_Merge[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Attention_GAP                 │ (None, 48)                │               0 │ Merge_BN[0][0]             │
│ (GlobalAveragePooling1D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Attention_Weights (Dense)     │ (None, 48)                │           2,352 │ Attention_GAP[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Attention_Reshape (Reshape)   │ (None, 1, 48)             │               0 │ Attention_Weights[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Attention_Apply (Multiply)    │ (None, 32, 48)            │               0 │ Merge_BN[0][0],            │
│                               │                           │                 │ Attention_Reshape[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Final_Pool (MaxPooling1D)     │ (None, 16, 48)            │               0 │ Attention_Apply[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Final_Flatten (Flatten)       │ (None, 768)               │               0 │ Final_Pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Dense_16 (Dense)              │ (None, 16)                │          12,304 │ Final_Flatten[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Dropout_1 (Dropout)           │ (None, 16)                │               0 │ Dense_16[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Dense_8_A (Dense)             │ (None, 8)                 │             136 │ Dropout_1[0][0]            │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 15,266 (59.63 KB)

 Trainable params: 15,170 (59.26 KB)

 Non-trainable params: 96 (384.00 B)

In [33]:
print("y_train type:", type(y_train))
print("y_train shape:", getattr(y_train, 'shape', 'N/A'))
print("y_train head / first 10 values:")
print(y_train.head() if hasattr(y_train, 'head') else y_train[:10])

print("\ny_test type:", type(y_test))
print("y_test shape:", getattr(y_test, 'shape', 'N/A'))
print("y_test head / first 10 values:")
print(y_test.head() if hasattr(y_test, 'head') else y_test[:10])


y_train type: <class 'pandas.core.series.Series'>
y_train shape: (40050489,)
y_train head / first 10 values:
27501869     DrDoS_UDP
26710292     DrDoS_UDP
26982585     DrDoS_UDP
49317128          TFTP
18594705    DrDoS_SNMP
Name: label, dtype: object

y_test type: <class 'pandas.core.series.Series'>
y_test shape: (10012623,)
y_test head / first 10 values:
36870664             TFTP
48513026             TFTP
48738044             TFTP
15511162    DrDoS_NetBIOS
9731615       DrDoS_MSSQL
Name: label, dtype: object


In [34]:
# Convert labels to binary: BENIGN -> 0, all attacks -> 1
y_train = (y_train != "BENIGN").astype(int)
y_test  = (y_test != "BENIGN").astype(int)

# Optional sanity check
print("y_train unique values:", y_train.unique())
print("y_test unique values:", y_test.unique())
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


y_train unique values: [1 0]
y_test unique values: [1 0]
y_train shape: (40050489,)
y_test shape: (10012623,)


In [ ]:
# 1. Compile the model
cnn_binary.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 2. Define the training callbacks
callbacks = [
    EarlyStopping(
        patience=5, 
        restore_best_weights=True, 
        monitor='val_accuracy'
    ),
    ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.5, 
        patience=5, 
        min_lr=1e-7, 
        verbose=1
    )
]

# 3. Start training
print("Starting CIAM Classifier Training...")
history_cnn = cnn_binary.fit(
    X_train_cnn, y_train,
    epochs=5,
    batch_size=1024,
    validation_data=(X_test_cnn, y_test),
    callbacks=callbacks,
    verbose=1
)

Starting CIAM Classifier Training...
Epoch 1/5
   54/39112 ━━━━━━━━━━━━━━━━━━━━ 18:48:15 2s/step - accuracy: 0.9691 - loss: 0.4291